# 🌬️ Wind Power Generation Prediction
**Target:** Predict `Power` output from meteorological features  
**Algorithm:** Random Forest Regressor  
**Dataset:** 175,200 rows × 4 Locations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded!')

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('merged_locations.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
print('Null values:\n', df.isnull().sum())
print('\nLocations:', df['Location'].unique())
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].hist(df['Power'], bins=50, color='steelblue', edgecolor='white')
axes[0,0].set_title('Power Output Distribution')

df.boxplot(column='Power', by='Location', ax=axes[0,1])
axes[0,1].set_title('Power by Location')

axes[1,0].scatter(df['windspeed_100m'], df['Power'], alpha=0.05, color='coral')
axes[1,0].set_xlabel('Wind Speed 100m'); axes[1,0].set_ylabel('Power')
axes[1,0].set_title('Wind Speed 100m vs Power')

axes[1,1].scatter(df['windgusts_10m'], df['Power'], alpha=0.05, color='green')
axes[1,1].set_xlabel('Wind Gusts 10m'); axes[1,1].set_ylabel('Power')
axes[1,1].set_title('Wind Gusts vs Power')

plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
numeric_cols = df.select_dtypes(include=[np.number]).columns
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.tight_layout(); plt.show()

## 3. Preprocessing — One-Hot Encode Location

In [ ]:
df = df.drop(columns=['Time'])
df = pd.get_dummies(df, columns=['Location'], drop_first=False)

TARGET = 'Power'
FEATURES = [c for c in df.columns if c != TARGET]

print('Features:', FEATURES)

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 4. Model Training & Comparison

In [ ]:
models = {
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Linear Regression': LinearRegression()
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    r2   = r2_score(y_test, preds)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    results[name] = {'R2': r2, 'MAE': mae, 'RMSE': rmse, 'model': model, 'preds': preds}
    print(f'{name:22s} | R²={r2:.4f} | MAE={mae:.4f} | RMSE={rmse:.4f}')

best_name = max(results, key=lambda k: results[k]['R2'])
print(f'\n✅ Best Model: {best_name} — R² = {results[best_name]["R2"]:.4f}')

In [ ]:
names     = list(results.keys())
r2_vals   = [results[n]['R2']   for n in names]
mae_vals  = [results[n]['MAE']  for n in names]
rmse_vals = [results[n]['RMSE'] for n in names]
x = np.arange(len(names))
colors = ['steelblue', 'coral', 'seagreen']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, vals, title in zip(axes,
    [r2_vals, mae_vals, rmse_vals],
    ['R² Score (higher=better)', 'MAE (lower=better)', 'RMSE (lower=better)']):
    bars = ax.bar(x, vals, color=colors)
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha='right')
    ax.set_title(title)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.4f}', ha='center', fontsize=9)
plt.tight_layout(); plt.show()

## 5. Best Model — Evaluation

In [ ]:
best_model = results[best_name]['model']
best_preds = results[best_name]['preds']

print(f'R²   : {results[best_name]["R2"]:.4f}')
print(f'MAE  : {results[best_name]["MAE"]:.4f}')
print(f'RMSE : {results[best_name]["RMSE"]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sample = np.random.choice(len(y_test), 2000, replace=False)

axes[0].scatter(np.array(y_test)[sample], best_preds[sample], alpha=0.3, color='steelblue', s=10)
axes[0].plot([0,1],[0,1],'r--', lw=2, label='Perfect fit')
axes[0].set_xlabel('Actual Power'); axes[0].set_ylabel('Predicted Power')
axes[0].set_title(f'Actual vs Predicted — R²={results[best_name]["R2"]:.4f}')
axes[0].legend()

residuals = np.array(y_test) - best_preds
axes[1].hist(residuals, bins=60, color='coral', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_xlabel('Residual'); axes[1].set_title('Residual Distribution')

plt.tight_layout(); plt.show()

In [ ]:
importances = best_model.feature_importances_
feat_df = pd.DataFrame({'Feature': FEATURES, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(feat_df['Feature'], feat_df['Importance'], color='steelblue')
plt.xlabel('Feature Importance')
plt.title('Random Forest — Feature Importances')
plt.tight_layout(); plt.show()

## 6. Save Model (.sav)

In [ ]:
joblib.dump(best_model, 'wind_power_model.sav')
print('✅ Saved: wind_power_model.sav')

# Verify
loaded = joblib.load('wind_power_model.sav')
print('🔁 Verify R²:', r2_score(y_test, loaded.predict(X_test)))

## 7. Sample Prediction

In [ ]:
sample = pd.DataFrame([{
    'temperature_2m': 28.5, 'relativehumidity_2m': 85, 'dewpoint_2m': 24.5,
    'windspeed_10m': 5.0, 'windspeed_100m': 8.0,
    'winddirection_10m': 180, 'winddirection_100m': 175, 'windgusts_10m': 10.0,
    'Location_Location1.csv': 1, 'Location_Location2.csv': 0,
    'Location_Location3.csv': 0, 'Location_Location4.csv': 0
}])[FEATURES]

pred = loaded.predict(sample)[0]
print(f'🌬️  Predicted Power Output: {pred:.4f}')